# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType
from pyspark.sql.window import Window
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.crm_products")

# Read from Bronze

In [0]:
try:
    df = spark.table("workspace.bronze.crm_prd_info_raw")
except Exception as e:
    logger.error(f"Failed to read Bronze table: {e}")
    raise

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Casting data types

In [0]:
df = (
    df.withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("product_cost", F.col("product_cost").cast(IntegerType()))
    .withColumn("start_date", F.col("start_date").cast(DateType()))
    .withColumn("end_date", F.col("end_date").cast(DateType()))
)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## Product Key Parsing

In [0]:
# add new column cat_id based on product_key substring of 5 characters ie. 'CO-RF' with '-' replace with '_' -> 'CO_RF'
df = df.withColumn("cat_id", F.regexp_replace(F.substring(F.col("product_key"), 1, 5), "-", "_"))
df = df.withColumn("product_key", F.substring(F.col("product_key"), 7, F.length(F.col("product_key"))))
df = df.withColumn("product_key_prefix", F.concat_ws("-", F.split(F.col("product_key"), "-")[0], F.split(F.col("product_key"), "-")[1]))

## Dedup and Cost Cleanup by product mean

In [0]:
window = Window.partitionBy("product_key_prefix")

df = df.dropDuplicates()
df = df.withColumn(
    "product_cost",
    F.coalesce(
        F.col("product_cost"), 
        F.round(F.avg("product_cost").over(window))
    )
)

## Product Line Normalization

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("product_line")) == "M", "Mountain")
        .when(F.upper(F.col("product_line")) == "R", "Road")
        .when(F.upper(F.col("product_line")) == "S", "Other Sales")
        .when(F.upper(F.col("product_line")) == "T", "Touring")
        .otherwise("Unknown")
    )
)

# Write into Silver Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")
except Exception as e:
    logger.error(f"Failed to write Silver table: {e}")
    raise